In [7]:
import sys
import os
import cv2
import supervision as sv
from ultralytics.models.sam import SAM3SemanticPredictor

# 1. Configurar rutas importando el archivo de configuración para obtener el modelo optimizado
sys.path.append(os.path.abspath("../src"))
from config import MODEL_PATH

# 2. Inicializar el predictor optimizado
print("Cargando modelo SAM 3...")
predictor = SAM3SemanticPredictor(
    overrides=dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model=MODEL_PATH,
        verbose=False,
        device="cpu" 
    )
)

# 3. Preparar el directorio de salida para las evidencias
output_dir = "../docs/assets/m1/prompts/"
os.makedirs(output_dir, exist_ok=True)

# 4. Definir las 5 pruebas requeridas: (Prompt, Frame de prueba)
pruebas = [
    ("robot", "frame_0050.jpg"),
    ("ball", "frame_0050.jpg"),
    ("soccer robot", "frame_0001.jpg"),
    ("player", "frame_0100.jpg"),
    ("person", "frame_0050.jpg")
]

# Herramientas de visualización de Supervision
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)

# 5. Ejecutar inferencia y guardar imágenes
for prompt, frame_file in pruebas:
    img_path = f"../docs/assets/m1/frames/{frame_file}"
    image = cv2.imread(img_path)
    
    if image is None:
        print(f"No se encontró la imagen {img_path}")
        continue
        
    print(f"Analizando prompt: '{prompt}'...")
    
    # Inferencia con SAM 3
    predictor.set_image(image)
    result = predictor(text=[prompt])[0]
    
    # Conversión al formato universal
    detections = sv.Detections.from_ultralytics(result)
    
    # Dibujar máscaras y etiquetas
    annotated = image.copy()
    annotated = mask_annotator.annotate(scene=annotated, detections=detections)
    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections,
        labels=[prompt] * len(detections)
    )
    
    # Guardar archivo con formato limpio
    nombre_archivo = f"prompt_{prompt.replace(' ', '_')}.jpg"
    out_path = os.path.join(output_dir, nombre_archivo)
    cv2.imwrite(out_path, annotated)
    print(f"Evidencia guardada: {out_path}\n")

print("Pruebas del M1-03 completadas")

Cargando modelo SAM 3...
Analizando prompt: 'robot'...
Ultralytics 8.4.57  Python-3.11.15 torch-2.12.0+cpu CPU (AMD Ryzen 5 5625U with Radeon Graphics)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]
Results saved to C:\Users\diego\OneDrive\Documentos\GitHub\Proyecto-VisionPorComputadora\notebooks\runs\segment\predict-7
Evidencia guardada: ../docs/assets/m1/prompts/prompt_robot.jpg

Analizando prompt: 'ball'...
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]
Results saved to C:\Users\diego\OneDrive\Documentos\GitHub\Proyecto-VisionPorComputadora\notebooks\runs\segment\predict-7
Evidencia guardada: ../docs/assets/m1/prompts/prompt_ball.jpg

Analizando prompt: 'soccer robot'...
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]
Results saved to C:\Users\diego\OneDrive\Documentos\GitHub\Proyecto-VisionPorComputadora\notebooks\runs\segment\predict-7
Evidencia guardada: ../docs/assets/m1/prompts/prompt_soccer_robot.jpg
